# Few-Shot Learning Lab: Prototype-Based Job-Scam Triage

Companion notebook for *AI for Security and Security for AI*.


## Goal

This notebook implements a simple few-shot prototype classifier for security text and shows how one mislabeled support example can damage performance.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


In [ ]:
DIFRAUD_BASE = "https://huggingface.co/datasets/difraud/difraud/resolve/main"

def load_difraud_domain(domain, splits=("train", "validation", "test")):
    frames = []
    for split in splits:
        url = f"{DIFRAUD_BASE}/{domain}/{split}.jsonl"
        tmp = pd.read_json(url, lines=True)
        tmp["split"] = split
        tmp["domain"] = domain
        frames.append(tmp)
    df = pd.concat(frames, ignore_index=True)
    df = df.dropna(subset=["text", "label"])
    df["text"] = df["text"].astype(str)
    df["label"] = df["label"].astype(int)
    return df

df = load_difraud_domain("job_scams")
train_df = df[df["split"] == "train"].copy()
test_df = df[df["split"] == "test"].copy()
print(train_df.shape, test_df.shape)
print(train_df["label"].value_counts())


## Build TF-IDF/SVD embeddings


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from sklearn.metrics import classification_report, accuracy_score

vectorizer = TfidfVectorizer(max_features=50000, ngram_range=(1, 2), min_df=2)
X_train_tfidf = vectorizer.fit_transform(train_df["text"])
svd = TruncatedSVD(n_components=100, random_state=RANDOM_STATE)
svd.fit(X_train_tfidf)

def embed(texts):
    X = vectorizer.transform(pd.Series(texts).astype(str))
    Z = svd.transform(X)
    return normalize(Z)


## Prototype classifier


In [ ]:
def make_support_set(data, k=5, random_state=RANDOM_STATE):
    pieces = []
    for label, group in data.groupby("label"):
        pieces.append(group.sample(n=min(k, len(group)), random_state=random_state))
    return pd.concat(pieces).sample(frac=1, random_state=random_state).reset_index(drop=True)

def build_prototypes(support_df):
    Z = embed(support_df["text"])
    y = support_df["label"].values
    prototypes = {}
    for c in sorted(np.unique(y)):
        proto = Z[y == c].mean(axis=0).reshape(1, -1)
        prototypes[int(c)] = normalize(proto)[0]
    return prototypes

def predict_with_prototypes(texts, prototypes):
    Z = embed(texts)
    classes = sorted(prototypes.keys())
    P = np.vstack([prototypes[c] for c in classes])
    sims = Z @ P.T
    pred_idx = np.argmax(sims, axis=1)
    preds = np.array([classes[i] for i in pred_idx])
    sorted_sims = np.sort(sims, axis=1)
    margins = sorted_sims[:, -1] - sorted_sims[:, -2]
    return preds, margins


In [ ]:
K = 5
support_df = make_support_set(train_df, k=K)
prototypes = build_prototypes(support_df)
pred, margin = predict_with_prototypes(test_df["text"], prototypes)
print(f"Clean support set, K={K} per class")
print(classification_report(test_df["label"], pred, zero_division=0))
print("Accuracy:", round(accuracy_score(test_df["label"], pred), 4))
support_df[["label", "text"]].head()


## Support-set noise experiment


In [ ]:
def flip_one_support_label(support_df, random_state=RANDOM_STATE):
    noisy = support_df.copy()
    rng = np.random.default_rng(random_state)
    idx = rng.choice(noisy.index.values, size=1)[0]
    noisy.loc[idx, "label"] = 1 - int(noisy.loc[idx, "label"])
    return noisy, idx

noisy_support, flipped_idx = flip_one_support_label(support_df)
noisy_prototypes = build_prototypes(noisy_support)
noisy_pred, noisy_margin = predict_with_prototypes(test_df["text"], noisy_prototypes)
print("After flipping one support-set label")
print("Flipped row:", flipped_idx)
print(classification_report(test_df["label"], noisy_pred, zero_division=0))
print("Accuracy:", round(accuracy_score(test_df["label"], noisy_pred), 4))


## Low-margin examples for analyst review


In [ ]:
triage = test_df.copy()
triage["prediction"] = pred
triage["margin"] = margin
triage["text_preview"] = triage["text"].str.slice(0, 240)
review_queue = triage.sort_values("margin").head(20)
review_queue[["label", "prediction", "margin", "text_preview"]]


## Practitioner takeaway

Few-shot classification is useful for rapid triage, but the support set is security-critical. Low-margin predictions should be routed to analyst review.
